In [3]:
import pandas as pd
import random

In [5]:
# Preparing the Data

train = pd.read_parquet("train.parquet")
val = pd.read_parquet("validation.parquet")
test = pd.read_parquet("test.parquet")

train = train.set_index("id")
val = val.set_index("id")
test = test.set_index("id")

train.drop(columns=["ner_tags"], inplace=True)
val.drop(columns=["ner_tags"], inplace=True)
test.drop(columns=["ner_tags"], inplace=True)

ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.

In [133]:
fine_ner_tag_dict = {
    0: "O",
    1: "art-broadcastprogram",
    2: "art-film",
    3: "art-music",
    4: "art-other",
    5: "art-painting",
    6: "art-writtenart",
    7: "building-airport",
    8: "building-hospital",
    9: "building-hotel",
    10: "building-library",
    11: "building-other",
    12: "building-restaurant",
    13: "building-sportsfacility",
    14: "building-theater",
    15: "event-attack/battle/war/militaryconflict",
    16: "event-disaster",
    17: "event-election",
    18: "event-other",
    19: "event-protest",
    20: "event-sportsevent",
    21: "location-GPE",
    22: "location-bodiesofwater",
    23: "location-island",
    24: "location-mountain",
    25: "location-other",
    26: "location-park",
    27: "location-road/railway/highway/transit",
    28: "organization-company",
    29: "organization-education",
    30: "organization-government/governmentagency",
    31: "organization-media/newspaper",
    32: "organization-other",
    33: "organization-politicalparty",
    34: "organization-religion",
    35: "organization-showorganization",
    36: "organization-sportsleague",
    37: "organization-sportsteam",
    38: "other-astronomything",
    39: "other-award",
    40: "other-biologything",
    41: "other-chemicalthing",
    42: "other-currency",
    43: "other-disease",
    44: "other-educationaldegree",
    45: "other-god",
    46: "other-language",
    47: "other-law",
    48: "other-livingthing",
    49: "other-medical",
    50: "person-actor",
    51: "person-artist/author",
    52: "person-athlete",
    53: "person-director",
    54: "person-other",
    55: "person-politician",
    56: "person-scholar",
    57: "person-soldier",
    58: "product-airplane",
    59: "product-car",
    60: "product-food",
    61: "product-game",
    62: "product-other",
    63: "product-ship",
    64: "product-software",
    65: "product-train",
    66: "product-weapon"
}

In [134]:
#sample random n fine ner tags

def fine_ner_tags(n):
    return [random.randint(1, 66) for _ in range(n)]

ner_tags = fine_ner_tags(2)

In [135]:
#sample k examples for each of the n fine ner tags

def sample_data_train(ner_tags, k, data, random_state=None):
    result = {}

    for tag in ner_tags:
        filtered = data[data["fine_ner_tags"].apply(lambda x: tag in x)]

        if len(filtered) < k:
            raise ValueError(f"Tag {tag} has only {len(filtered)} rows, but k={k}")

        sampled = filtered.sample(n=k, random_state=random_state).copy()

        sampled["fine_ner_tags"] = sampled["fine_ner_tags"].apply(
            lambda labels: [label if label == tag else 0 for label in labels]
        )
        
        result[tag] = sampled


    return result

In [136]:
# Select and prepare all the data with the ner tags for validation and testing

def sample_data_test(ner_tags, data):
    result = {}

    for tag in ner_tags:
        sampled = data[data["fine_ner_tags"].apply(lambda x: tag in x)]

        sampled["fine_ner_tags"] = sampled["fine_ner_tags"].apply(
            lambda labels: [label if label == tag else 0 for label in labels]
        )
        
        result[tag] = sampled

    return result

In [137]:
# Example usage

print(fine_ner_tag_dict[ner_tags[0]])
sample = sample_data_train(ner_tags, 5, train)

s = sample[ner_tags[0]]

print(s.head())


validation = sample_data_test(ner_tags, val)


art-film
                                                   tokens  \
id                                                          
28463   [It, is, second, time, team, up, for, Jagapath...   
90812   [He, left, the, company, during, Disney, anima...   
105351  [His, acting, roles, include, ``, Stickmen, ``...   
93190   [The, film, is, a, remake, of, Telugu, film, `...   
17968   [While, Diana, enjoys, London, society, ,, Mat...   

                                            fine_ner_tags  
id                                                         
28463   [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...  
90812   [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...  
105351  [0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...  
93190   [0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, ...  
17968   [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...  
